# Dupin initialization

Creation of Dupin L-meshes from propagation.

## Initialize

Add the project root so `LMesh` and `Lconjugacy` import (hanan and kayviz are installed). Adjust `PROJECT_ROOT` if the notebook is moved.

In [ ]:
import sys, os
import numpy as np
from hanan.geometry.utils import (read_obj, 
                                  normalize_vertices,
                                  compute_cone_axes
                                 )

from hanan.geometry.mesh import Mesh


# Load project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

from LMesh.L_mesh import Lmesh
import kayviz as kv




DATA = os.path.join(PROJECT_ROOT, "data")

# Verification of for Euclidean lines

In [ ]:
from hanan.geometry.utils import (circle_3pts, circle_3d, merge_curves)

def compute_intersection_distance_curve(basemesh, axes, angles, curves):
    """
    Compute the radii for each isoparameter family of the conical mesh.
    """

    from hanan.geometry.utils import clst_point_lines

    dist = np.zeros(len(basemesh.vertices))

    for i, curve in enumerate(curves):

        for j in range(len(curve) - 1):

            id1 = curve[j]
            id2 = curve[j + 1]

            p1 = basemesh.vertices[id1]
            p2 = basemesh.vertices[id2]
            
            d1 = axes[id1]
            d2 = axes[id2]


            # Compute the closest points on the first line
            int_1, _, _ = clst_point_lines(p1, d1, p2, d2)

            # Store the distance as the radius for both points
            dist[id1] =  np.sign( (int_1 - p1) @ d1 )  *   np.linalg.norm(int_1 - p1) 
            
    return dist



def compute_dupin_pts(basemesh, axes, angles):
    """
    Compute the radii for each isoparameter family of the conical mesh.
    """

    from hanan.geometry.utils import clst_point_lines, plane_plane_intersection, unit

    # Extract u, v isocurves of the quad mesh
    u_curves, v_curves = basemesh.quad_iso_curves()

    dupin_pts = {}

    f_n = basemesh.face_normals

    inn_v = basemesh.inner_vertices()

    tau = 1

    dist_u = tau * compute_intersection_distance_curve(basemesh, axes, angles, u_curves)
    dist_v = tau * compute_intersection_distance_curve(basemesh, axes, angles, v_curves)

    # Where dist_u and dist_v have the same sign, they must agree — collapse
    # both to the smaller-magnitude value. Opposite-sign rows are left as is.
    same_sign = (np.sign(dist_u) == np.sign(dist_v))
    smaller   = np.where(np.abs(dist_u) < np.abs(dist_v), dist_u, dist_v)
    dist_u = np.where(same_sign, smaller, dist_u)
    dist_v = np.where(same_sign, smaller, dist_v)


    u_adj, v_adj = basemesh.quad_vertex_adjacency()


    intersection_lines = {}
    

    for i in inn_v:

        v0 = basemesh.vertices[i]
        d = axes[i]

        cu_0 = v0 + dist_u[i] * d
        cv_0 = v0 + dist_v[i] * d

        # Get the adjacent vertices in the u and v directions
        for direction, adj, c0, dist in (("u", u_adj[i], cv_0, dist_v), ("v", v_adj[i], cu_0, dist_u)):
            for id_n in adj:
                f1, f2 = basemesh.faces_from_edge(i, id_n)

                # Get normals faces
                n1 = f_n[f1]
                n2 = f_n[f2]

                v_n = basemesh.vertices[id_n]

                c_n = v_n + dist[id_n] * axes[id_n]

                cn = unit(c_n - c0)

                # Compute the normal of the plane that contain cn and n1
                P_n1 = unit(np.cross(cn, n1))
                P_n2 = unit(np.cross(cn, n2))

                # Get the intersection  P_1 \cap f_1
                l1 = plane_plane_intersection(P_n1, c0, n1, v0)
                l2 = plane_plane_intersection(P_n2, c0, n2, v0)

                intersection_lines.setdefault((i, f1), []).append(l1)
                intersection_lines.setdefault((i, f2), []).append(l2)

    # Each (vertex_id, face) key must collect exactly two lines (one from the
    # u-direction neighbor, one from the v-direction neighbor) — intersect them.
    for (vid, fid), lines in intersection_lines.items():
        assert len(lines) == 2, f"expected 2 lines for {(vid, fid)}, got {len(lines)}"

        (p1, dir1), (p2, dir2) = lines
        _, _, m = clst_point_lines(p1, dir1, p2, dir2)

        dupin_pts.setdefault(vid, []).append(m)

    return dupin_pts

meshname = "cone_mesh_0.obj"
#meshname = "hexMeshTest.obj"
#meshname = "seashell_optimized.obj"
#meshname = "lilium.obj"


mesh_path = os.path.join(DATA, meshname)
V, F = read_obj(mesh_path)
V    = normalize_vertices(V)
print(f"Vertices: {len(V)}   Faces: {len(F)}")

# Visualize mesh
kv.clear()   # remove anything left from a previous run
kv.register_surface_mesh("Base Mesh", V, F, color=(0.2, 0.2, 0.8), show_edges=True, edge_color=(1,1,1), opacity=0.5)
kv.show() 


basemesh = Mesh()
basemesh.make_mesh(V, F)
in_v = basemesh.inner_vertices()

axes, angles = compute_cone_axes(basemesh)

dupin_pts = compute_dupin_pts(basemesh, axes, angles)


dupin_at_vertex = dupin_pts[in_v[270]] 

# circle_v0 = circle_3pts(dupin_at_vertex[0], dupin_at_vertex[1], dupin_at_vertex[2])
# circlle_draw = circle_3d(circle_v0[0], circle_v0[1], circle_v0[2])

# edges_circle = np.array([[i, (i + 1) % len(circlle_draw)] for i in range(len(circlle_draw))])

circles = []

for i, v_id in enumerate(in_v):
    dupin_at_vertex = dupin_pts[v_id] 
    circle_v0 = circle_3pts(dupin_at_vertex[0], dupin_at_vertex[1], dupin_at_vertex[2])
    circlle_draw = circle_3d(circle_v0[0], circle_v0[1], circle_v0[2])

    edges_circle = np.array([[i, (i + 1) % len(circlle_draw)] for i in range(len(circlle_draw))])

    circles.append(circlle_draw)


circle_merge, edge_merge = merge_curves(circles)

kv.register_point_cloud("Dupin Poits", list(dupin_pts.values()), color=(1, 0, 0), radius=0.001)
kv.register_curve_network("Dupin Circle", circle_merge, edge_merge, color=(0, 1, 0), radius=0.0002)

scaled_axes = axes / np.sin(angles[:, None])


# radii = 0.5 * smallest_radii

# centers = V + radii[:, np.newaxis] * scaled_axes

# lmesh = Lmesh()
# lmesh.init_L_mesh(
#     basemesh,
#     orientation=1,
#     combinatorics="primal",
#     mode="SphCongruence",
#     radii=radii,
#     centers=centers,
# )


# lmesh.improve_contact()


# Analysis

In [ ]:
from hanan.geometry.utils import clst_point_lines, plane_plane_intersection, unit, clip_line_to_convex_face
# Extract u, v isocurves of the quad mesh
u_curves, v_curves = basemesh.quad_iso_curves()

dupin_pts = {}

f_n = basemesh.face_normals

inn_v = basemesh.inner_vertices()

tau = 1

dist_u = tau * compute_intersection_distance_curve(basemesh, axes, angles, u_curves)
dist_v = tau * compute_intersection_distance_curve(basemesh, axes, angles, v_curves)
same_sign = (np.sign(dist_u) == np.sign(dist_v))
smaller   = np.where(np.abs(dist_u) < np.abs(dist_v), dist_u, dist_v)
dist_u = np.where(same_sign, smaller, dist_u)
dist_v = np.where(same_sign, smaller, dist_v)


u_adj, v_adj = basemesh.quad_vertex_adjacency()


intersection_lines = {}
for i in inn_v:

    v0 = basemesh.vertices[i]
    d = axes[i]

    cu_0 = v0 + dist_u[i] * d
    cv_0 = v0 + dist_v[i] * d

    # Get the adjacent vertices in the u and v directions
    for direction, adj, c0, dist in (("u", u_adj[i], cv_0, dist_v), ("v", v_adj[i], cu_0, dist_u)):
        for id_n in adj:
            f1, f2 = basemesh.faces_from_edge(i, id_n)

            # Get normals faces
            n1 = f_n[f1]
            n2 = f_n[f2]

            v_n = basemesh.vertices[id_n]

            c_n = v_n + dist[id_n] * axes[id_n]

            cn = unit(c_n - c0)

            # Compute the normal of the plane that contain cn and n1
            P_n1 = unit(np.cross(cn, n1))
            P_n2 = unit(np.cross(cn, n2))

            # Get the intersection  P_1 \cap f_1
            l1 = plane_plane_intersection(P_n1, c0, n1, v0)
            l2 = plane_plane_intersection(P_n2, c0, n2, v0)

            intersection_lines.setdefault((i, f1), []).append(l1)
            intersection_lines.setdefault((i, f2), []).append(l2)


vertex_face_adj = basemesh.vertex_face_adjacency_list()
v_adj = basemesh.vertex_adjacency_list()


vertex_id = 289

print(dist_u[vertex_id], dist_v[vertex_id])
for vj in v_adj[vertex_id]:
    print(dist_u[vj], dist_v[vj])
    

#kv.add_vector_quantity("Base Mesh", "Axes", axes)

for face in vertex_face_adj[vertex_id]:
    lines = intersection_lines[(vertex_id, face)]
    face_vertices = basemesh.vertices[basemesh.faces[face]]
    kv.register_surface_mesh(f"Face {face}", face_vertices, [np.arange(len(face_vertices))], color=(1, 0, 0), opacity=0.5)

    for li, line in enumerate(lines):
        p, d = line
        result = clip_line_to_convex_face(p, d, basemesh.vertices[basemesh.faces[face]])
        
        if result[0] is not None and result[1] is not None:
            tm, tM = result

            p0 = p + tm * d
            p1 = p + tM * d

            kv.register_curve_network(f"Vertex {vertex_id}, Face {face} line {li}", [p0, p1], [[0, 1]], color=(1, 1, 1), radius=0.0002)
        else: 
            

            print(f"Line does not intersect face {face} for vertex {vertex_id}")
            